In [1]:
import pandas as pd
import numpy as np
from numpy.random import Generator, PCG64
from joblib import Parallel, delayed
import altair as alt

rng = Generator(PCG64())

In [2]:
all_df = pd.read_csv("./dataframes/all_evaluated_responses_df.csv")
all_df = all_df.dropna(subset=['agent_response'])
all_df['item_id'] = all_df['question'] + all_df['image_file']

In [3]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

# Create Item DF

### Chart Type Labels

In [4]:
chart_type_maps = {
    'Line Chart': "line", 
    'Pie Chart': "pie", 
    'Bar Chart': "bar", 
    'Dot Plot': "dot-plot", 
    'Scatterplot': 'scatter',
    'Bubble Chart': 'bubble', 
    'Stacked Bar Chart': "stacked-bar", 
    'Stacked Area Chart': "stacked-area",
    'Treemap': 'treemap', 
    'Histogram': 'histogram', 
    'Choropleth Map': 'map', 
    'Area Chart': 'area',
    '100 % Stacked Bar Chart': '100-stacked-bar', 
    'Stacked area chart': 'stacked-area', 
    'Bar chart': 'bar',
    'Stacked bar chart': 'stacked-bar', 
    'Area chart': 'area', 
    'Choropleth map': 'map', 
    'Line chart': 'line',
    'Pie chart': 'pie', 
    '100% stacked bar chart': '100-stacked-bar',
    'h_bar_PEW': 'bar',
    'v_bar_two_var': 'bar', 
    'h_bar_two_var': 'bar', 
    'pie_PEW': 'pie', 
    'v_bar_multi_var': 'bar',
    'line_two_var': 'line', 
    'line_multi_var': 'line', 
    'pie_Statista': 'pie',
    'h_bar_multi_var': 'bar', 
    'line_OWID': 'line',
    'barchart': 'bar',
}

### Task Type Labels

In [5]:
vlat_format_map = {
    'find_extremum': 'value identification', 
    'determine_Range': 'value identification', 
    'retrieve_value': 'value identification',
    'make_comparisons': 'arithmetic computation',
    'find_correlations_trends': 'statistical inference', 
    'characterize_distribution': 'statistical inference',
    'find_anomolies': 'statistical inference', 
    'find_clusters': 'statistical inference', 
}

calvi_format_map = {
    'Make Comparisons': 'arithmetic computation',
    'Find Correlations/Trends': 'statistical inference', 
    'Find Extremum': 'value identification',
    'Retrieve Value': 'value identification', 
    'Make Predictions': 'statistical inference', 
    'Aggregate Values': 'statistical inference'
}

ggr_question_map = {
    # from level 1
    "Approximately what percentage of people had Adeolitis in the year 2000?": "value identification",
    "What percentage of patients recovered after chemotherapy?": "value identification",
    "Of 100 patients with disease X, how many are women?": "value identification",
    "Of all the people who die from cancer, approximately what percentage dies from lung cancer?": "value identification",

    # from level 2
    "Approximately what percentage of people who die from cancer die from colon cancer, breast cancer, and prostate cancer taken together?": "arithmetic computation",
    "How many more men than women are there among 100 patients with disease X?": "arithmetic computation",
    "What is the difference between the percentage of patients who recovered after a surgery and the percentage of patients who recovered after radiation therapy?": "arithmetic computation",
    "When was the increase in the percentage of people with Adeolitis higher?": "arithmetic computation",

    # from level 3
    "What percentage of patients recovered after chemotherapy?": "value identification",
    "According to your best guess, what will the percentage of people with Adeolitis be in the year 2010?": "statistical inference",
    "Between 1980 and 1990, which disease had a higher increase in the percentage of people affected?": "statistical inference",
    "Compared to the placebo, which treatment leads to a larger decrease in the percentage of patients who die?": "statistical inference",
    "What is the percentage of cancer patients who die after chemotherapy?": "value identification",
    "Which of the treatments contributes to a larger decrease in the percentage of sick patients?": "statistical inference"
}

holf_map = {
    'retrieve_value': 'value identification', 
    'make_comparisons': 'arithmetic computation',
    'determine_range': 'arithmetic computation',
}

holf2_map = {
    'retrieve_value': 'value identification', 
    'make_comparisons': 'arithmetic computation',
    'predict_trend': 'statistical inference'
}

chartqa_map = {
    'retrive value': 'value identification'
}


task_category_maps = vlat_format_map | calvi_format_map | holf_map | holf2_map | chartqa_map

### Create item property dataframe

In [6]:
def create_item_df():
    tests = [
        'ggr', 
        'vlat', 
        'calvi', 
        'holf', 
        'holf2',
        'chartqa-test-continuous-human'
    ]
    
    item_df = []
    for test in tests:
        idf = pd.read_csv(f'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test}/questions.csv')
        idf['test_type'] = test

        if test == 'ggr':
            idf['task_category'] = idf['question'].replace(ggr_question_map)
        else:
            idf['task_category'] = idf['task_category'].replace(task_category_maps)

        item_df.append(idf)

    item_df = pd.concat(item_df)
    item_df['question_image'] = item_df['question'] + " + " + item_df['image_file']
    return item_df

item_df = create_item_df()
item_df['item_id'] = item_df['question'] + item_df['image_file']
item_df['chart_type'] = item_df['chart_type'].replace(chart_type_maps)

print(item_df['chart_type'].unique())
print(item_df['task_category'].unique())

['bar' 'line' 'pie' 'dot-plot' 'stacked-bar' '100-stacked-bar' 'histogram'
 'scatter' 'area' 'stacked-area' 'bubble' 'map' 'treemap']
['value identification' 'statistical inference' 'arithmetic computation']


## Feature Pairwise Figure

In [7]:
# group_average_df = all_df.groupby(['agent_type', 'test_type', 'item_id'])['is_correct'].mean().reset_index()
group_average_df = all_df[all_df['agent_type'].isin(['GPT-4V', 'Human'])]
group_average_df = pd.merge(group_average_df, item_df[['item_id', 'task_category', 'chart_type']])
group_average_df['test_type'] = group_average_df['test_type'].replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_df = group_average_df.groupby(['agent_type', 'test_type', 'task_category'])['is_correct'].mean().reset_index()
graph_group_average_df = group_average_df.groupby(['agent_type', 'test_type', 'chart_type'])['is_correct'].mean().reset_index()

In [8]:
# task_group_average_df

### Pairwise Slope Plots

In [9]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
graph_group_average_simplified = graph_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

graph_group_average_simplified = graph_group_average_simplified.groupby(['test_type', 'agent_type', 'chart_type'])['is_correct'].mean().reset_index()

for test in ['mc', 'num']:
    df = graph_group_average_simplified[graph_group_average_simplified['test_type'] == test]
    
    line_chart = alt.Chart(df).mark_line(opacity=0.8).encode(
        x=alt.X('agent_type:N', title=None),
        y=alt.Y('is_correct', scale=alt.Scale(domain=[0, 1]), title=None),
        detail='chart_type',
        # color=alt.value('black'),
        color=alt.Color('chart_type', scale=alt.Scale(scheme='category20'))
    ).properties(width=150, title=test)

    point_plot = alt.Chart(df).mark_circle(opacity=0.6).encode(
        x='agent_type:N',
        y='is_correct',
        # color=alt.Color('agent_type',legend=None, scale=alt.Scale(domain=domain, range=color_domain)),
        # color=alt.Color('test_type',legend=None, scale=alt.Scale(domain=test_domain, range=color_domain)),
        size=alt.value(40)
    )

    chart = (line_chart + point_plot).resolve_scale(color='shared').properties(width=100, height=200)

    charts.append(chart)

alt.hconcat(*charts)

alt.HConcatChart(...)

In [10]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
task_group_average_simplified = task_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_simplified = task_group_average_simplified.groupby(['test_type', 'agent_type', 'task_category'])['is_correct'].mean().reset_index()

for test in ['mc', 'num']:
    df = task_group_average_simplified[task_group_average_simplified['test_type'] == test]
    
    line_chart = alt.Chart(df).mark_line(opacity=0.8).encode(
        x=alt.X('agent_type:N', title=None),
        y=alt.Y('is_correct', scale=alt.Scale(domain=[0, 1]), title=None),
        detail='task_category',
        # color=alt.value('black'),
        color=alt.Color('task_category')
    ).properties(width=150, title=test)

    point_plot = alt.Chart(df).mark_circle(opacity=0.6).encode(
        x='agent_type:N',
        y='is_correct',
        # color=alt.Color('agent_type',legend=None, scale=alt.Scale(domain=domain, range=color_domain)),
        # color=alt.Color('test_type',legend=None, scale=alt.Scale(domain=test_domain, range=color_domain)),
        size=alt.value(40)
    )

    chart = (line_chart + point_plot).resolve_scale(color='shared').properties(width=100, height=200)

    charts.append(chart)

alt.hconcat(*charts)

alt.HConcatChart(...)

### Graph Group

In [11]:
# graph_group_average_df[
#     graph_group_average_df['agent_type'] == 'Human'
# ]

In [12]:
icon_graph_maps = pd.DataFrame([
    {'chart_type': '100-stacked-bar', 'icon': 'norm_stacked_bar.svg'}, 
    {'chart_type': 'area', 'icon': 'area.svg'}, 
    {'chart_type': 'bar', 'icon': 'bar.svg'}, 
    {'chart_type': 'bubble', 'icon': 'bubble-1-1.png'}, 
    {'chart_type': 'dot-plot', 'icon': 'dot-plot-1.svg'},
    {'chart_type': 'histogram', 'icon': 'histogram-1.png'}, 
    {'chart_type': 'line', 'icon': 'line.svg'}, 
    {'chart_type': 'map', 'icon': 'map.svg'}, 
    {'chart_type': 'pie', 'icon': 'pie.svg'}, 
    {'chart_type': 'scatter', 'icon': 'scatter.svg'}, 
    {'chart_type': 'stacked-area', 'icon': 'stacked_area.svg'},
    {'chart_type': 'stacked-bar', 'icon': 'stacked_bar.svg'}, 
    {'chart_type': 'treemap', 'icon': 'treemap-1.svg'}
])

icon_graph_maps['icon'] =  "https://raw.githubusercontent.com/vermaarn/img_store/main/" + icon_graph_maps['icon']
icon_graph_maps['icon'] = icon_graph_maps['icon'].apply(lambda l : l.replace('.svg', '.png'))

In [13]:
icon_graph_maps = pd.DataFrame([
    {'chart_type': '100-stacked-bar', 'icon': '100_stacked_bar_circle.png'}, 
    {'chart_type': 'area', 'icon': 'area_circle.png'}, 
    {'chart_type': 'bar', 'icon': 'bar_circle.png'}, 
    {'chart_type': 'bubble', 'icon': 'bubble_circle.png'}, 
    {'chart_type': 'dot-plot', 'icon': 'dot_circle.png'},
    {'chart_type': 'histogram', 'icon': 'histogram_circle.png'}, 
    {'chart_type': 'line', 'icon': 'line_circle.png'}, 
    {'chart_type': 'map', 'icon': 'map_circle.png'}, 
    {'chart_type': 'pie', 'icon': 'pie_circle.png'}, 
    {'chart_type': 'scatter', 'icon': 'scatter_circle.png'}, 
    {'chart_type': 'stacked-area', 'icon': 'stacked_area_circle.png'},
    {'chart_type': 'stacked-bar', 'icon': 'stacked_bar_circle.png'}, 
    {'chart_type': 'treemap', 'icon': 'treemap_circle.png'}
])

icon_graph_maps['icon'] =  "https://raw.githubusercontent.com/vermaarn/img_store_2/main/" + icon_graph_maps['icon']

In [14]:
# icon_graph_maps.iloc[0]['icon']
# icon_graph_maps

In [15]:
def create_graph_dataframe(graph_group_average_df):
    mc_graphs = graph_group_average_df[
        graph_group_average_df['test_type'] == 'mc'
    ].pivot(index='chart_type',columns=['agent_type'], values='is_correct').reset_index()
    mc_graphs['test_type'] = 'mc'
    
    num_graphs = graph_group_average_df[
        graph_group_average_df['test_type'] == 'num'
    ].pivot(index='chart_type',columns=['agent_type'], values='is_correct').reset_index()
    num_graphs['test_type'] = 'num'
    
    graph_group = pd.concat([mc_graphs, num_graphs])
    graph_group = pd.merge(graph_group, icon_graph_maps)
    return graph_group

In [16]:
graph_group = create_graph_dataframe(graph_group_average_df)

def update_icon_test_type(r):
    if r['test_type'] == 'mc':
        return r['icon']
    elif r['test_type'] == 'num':
        return r['icon'].replace('circle', 'square')

graph_group['icon'] = graph_group.apply(update_icon_test_type, axis=1)

In [17]:
# graph_group
graph_group

,chart_type,GPT-4V,Human,test_type,icon
0,100-stacked-bar,0.540541,0.748384,mc,https://raw.githubusercontent.com/vermaarn/img...
1,area,0.508772,0.619120,mc,https://raw.githubusercontent.com/vermaarn/img...
2,bar,0.427586,0.699078,mc,https://raw.githubusercontent.com/vermaarn/img...
3,bubble,0.771429,0.740977,mc,https://raw.githubusercontent.com/vermaarn/img...
4,dot-plot,0.000000,0.824078,mc,https://raw.githubusercontent.com/vermaarn/img...
5,histogram,0.533333,0.945778,mc,https://raw.githubusercontent.com/vermaarn/img...
6,line,0.591463,0.755970,mc,https://raw.githubusercontent.com/vermaarn/img...
7,map,0.305263,0.631241,mc,https://raw.githubusercontent.com/vermaarn/img...
8,pie,0.451220,0.834994,mc,https://raw.githubusercontent.com/vermaarn/img...
9,scatter,0.292683,0.673747,mc,https://raw.githubusercontent.com/vermaarn/img...


In [18]:

icon_width = 20
gpt_human_graph_performance_img_icon = alt.Chart(graph_group).mark_image(width=icon_width, height=icon_width).encode(
    x=alt.X('Human', scale=alt.Scale(domain=[0,1])),
    y=alt.Y('GPT-4V', scale=alt.Scale(domain=[0,1])),
    url=alt.Url('icon'),
    opacity=alt.value(1)
).properties(height=300, width=300)

line_layer = alt.Chart(
    pd.DataFrame({'x': [0, 1], 'y': [0, 1]})
).mark_line(strokeWidth=2, strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q',
    color=alt.value('black'),
    opacity=alt.value(0.3)
)

graph_performance = gpt_human_graph_performance_img_icon
graph_performance += line_layer
graph_performance.save("./figures/graph_feature_performance.pdf")
graph_performance

alt.LayerChart(...)

### Task Group

In [19]:
domain = ['GPT-4V', 'Human']
color_domain = [model_configs[-2]['color'], model_configs[-1]['color']]

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']

charts =[]
task_group_average_simplified = task_group_average_df.copy().replace({
    'ggr': 'mc', 
    'vlat': 'mc', 
    'calvi': 'mc',
    'holf': 'num', 
    'holf2': 'num', 
    'chartqa-test-continuous-human': 'num'
})

task_group_average_simplified = task_group_average_simplified.groupby(['test_type', 'agent_type', 'task_category'])['is_correct'].mean().reset_index()

icon_task_maps = pd.DataFrame([
    {'task_category': 'arithmetic computation', 'icon': 'arithmetic_sc_circle.png'}, 
    {'task_category': 'statistical inference', 'icon': 'inference_sc_circle.png'}, 
    {'task_category': 'value identification', 'icon': 'value_id_sc_circle.png'}, 
])
icon_task_maps['icon'] =  "https://raw.githubusercontent.com/vermaarn/img_store_2/main/" + icon_task_maps['icon']

In [20]:
def create_task_dataframe(task_group_average_simplified):
    mc_tasks = task_group_average_simplified[
        task_group_average_simplified['test_type'] == 'mc'
    ].pivot(index='task_category',columns=['agent_type'], values='is_correct').reset_index()
    mc_tasks['test_type'] = 'mc'
    
    num_tasks = task_group_average_simplified[
        task_group_average_simplified['test_type'] == 'num'
    ].pivot(index='task_category',columns=['agent_type'], values='is_correct').reset_index()
    num_tasks['test_type'] = 'num'

    task_group = pd.concat([mc_tasks, num_tasks])
    task_group = pd.merge(task_group, icon_task_maps)

    return task_group

In [21]:
task_group = create_task_dataframe(task_group_average_simplified)

def update_task_icon_test_type(r):
    if r['test_type'] == 'mc':
        return r['icon']
    elif r['test_type'] == 'num':
        return r['icon'].replace('circle', 'square')

task_group['icon'] = task_group.apply(update_task_icon_test_type, axis=1)

In [22]:
# graph_group_average_df

In [42]:
task_figure_icon_size = 20
task_pairwise_figure = alt.Chart(task_group).mark_image(width=task_figure_icon_size, height=task_figure_icon_size).encode(
    x=alt.X('Human', scale=alt.Scale(domain=[0,1])),
    y=alt.Y('GPT-4V', scale=alt.Scale(domain=[0,1])),
    url=alt.Url('icon'),
    opacity=alt.value(0.8),
    color=alt.value('black')
).properties(height=300, width=300)

line_layer = alt.Chart(
    pd.DataFrame({'x': [0, 1], 'y': [0, 1]})
).mark_line(strokeWidth=2, strokeDash=[5, 5]).encode(
    x='x:Q',
    y='y:Q',
    color=alt.value('black'),
    opacity=alt.value(0.3)
)
task_pairwise_figure += line_layer
task_pairwise_figure.save("./figures/task_feature_performance.pdf")

task_pairwise_figure

alt.LayerChart(...)

In [24]:
# icon_task_maps.iloc[0]['icon']

In [25]:
graph_group['gpt_human_diff'] = graph_group['Human'] - graph_group['GPT-4V']
graph_group.sort_values(by='gpt_human_diff')

,chart_type,GPT-4V,Human,test_type,icon,gpt_human_diff
10,stacked-area,0.723077,0.522787,mc,https://raw.githubusercontent.com/vermaarn/img...,-0.200290
16,pie,1.000000,0.875000,num,https://raw.githubusercontent.com/vermaarn/img...,-0.125000
3,bubble,0.771429,0.740977,mc,https://raw.githubusercontent.com/vermaarn/img...,-0.030451
15,line,0.230056,0.265137,num,https://raw.githubusercontent.com/vermaarn/img...,0.035082
17,scatter,0.052632,0.104027,num,https://raw.githubusercontent.com/vermaarn/img...,0.051395
14,bar,0.306304,0.365783,num,https://raw.githubusercontent.com/vermaarn/img...,0.059479
1,area,0.508772,0.619120,mc,https://raw.githubusercontent.com/vermaarn/img...,0.110348
18,stacked-bar,0.419355,0.553846,num,https://raw.githubusercontent.com/vermaarn/img...,0.134491
12,treemap,0.733333,0.892729,mc,https://raw.githubusercontent.com/vermaarn/img...,0.159396
6,line,0.591463,0.755970,mc,https://raw.githubusercontent.com/vermaarn/img...,0.164506


In [26]:
task_group['gpt_human_diff'] = task_group['Human'] - task_group['GPT-4V']
task_group.sort_values(by='gpt_human_diff')

,task_category,GPT-4V,Human,test_type,icon,gpt_human_diff
3,arithmetic computation,0.121278,0.151286,num,https://raw.githubusercontent.com/vermaarn/img...,0.030008
4,statistical inference,0.100877,0.153016,num,https://raw.githubusercontent.com/vermaarn/img...,0.052139
5,value identification,0.473128,0.535868,num,https://raw.githubusercontent.com/vermaarn/img...,0.062740
1,statistical inference,0.528249,0.715855,mc,https://raw.githubusercontent.com/vermaarn/img...,0.187606
0,arithmetic computation,0.400568,0.667544,mc,https://raw.githubusercontent.com/vermaarn/img...,0.266976
2,value identification,0.442424,0.744359,mc,https://raw.githubusercontent.com/vermaarn/img...,0.301934


#### Graph type correlation

In [27]:
np.corrcoef(graph_group['Human'], graph_group['GPT-4V'])

array([[1.        , 0.39855359],
       [0.39855359, 1.        ]])

In [28]:
np.corrcoef(task_group['Human'], task_group['GPT-4V'])

array([[1.        , 0.94115517],
       [0.94115517, 1.        ]])

## Calculate interaction

### Load R

In [29]:
%load_ext rpy2.ipython

In [30]:
%%R
install.packages("lme4")
library(lme4)

--- Please select a CRAN mirror for use in this session ---
Secure CRAN mirrors 

 1: 0-Cloud [https]
 2: Australia (Canberra) [https]
 3: Australia (Melbourne 1) [https]
 4: Australia (Melbourne 2) [https]
 5: Austria (Wien 1) [https]
 6: Belgium (Brussels) [https]
 7: Brazil (PR) [https]
 8: Brazil (SP 1) [https]
 9: Brazil (SP 2) [https]
10: Bulgaria [https]
11: Canada (MB) [https]
12: Canada (ON 1) [https]
13: Canada (ON 2) [https]
14: Chile (Santiago) [https]
15: China (Beijing 1) [https]
16: China (Beijing 2) [https]
17: China (Beijing 3) [https]
18: China (Hefei) [https]
19: China (Hong Kong) [https]
20: China (Jinan) [https]
21: China (Lanzhou) [https]
22: China (Nanjing) [https]
23: China (Shanghai 2) [https]
24: China (Shenzhen) [https]
25: China (Wuhan) [https]
26: Colombia (Cali) [https]
27: Costa Rica [https]
28: Cyprus [https]
29: Czech Republic [https]
30: Denmark [https]
31: East Asia [https]
32: Ecuador (Cuenca) [https]
33: Finland (Helsinki) [https]
34: France (Lyon 1

Selection:  68



The downloaded binary packages are in
	/var/folders/v8/3zpbxkws53b3x6m8509jyml80000gn/T//RtmpuiV4Xn/downloaded_packages


trying URL 'https://mirror.las.iastate.edu/CRAN/bin/macosx/big-sur-arm64/contrib/4.4/lme4_1.1-36.tgz'
Content type 'application/x-gzip' length 7079300 bytes (6.8 MB)
downloaded 6.8 MB

Loading required package: Matrix
In addition: Warning message:
In doTryCatch(return(expr), name, parentenv, handler) :
  unable to load shared object '/Library/Frameworks/R.framework/Resources/modules//R_X11.so':
  dlopen(/Library/Frameworks/R.framework/Resources/modules//R_X11.so, 0x0006): Library not loaded: /opt/X11/lib/libSM.6.dylib
  Referenced from: <34C5A480-1AC4-30DF-83C9-30A913FC042E> /Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/modules/R_X11.so
  Reason: tried: '/opt/X11/lib/libSM.6.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/X11/lib/libSM.6.dylib' (no such file), '/opt/X11/lib/libSM.6.dylib' (no such file), '/usr/local/lib/libSM.6.dylib' (no such file), '/usr/lib/libSM.6.dylib' (no such file, not in dyld cache)


### Load 

In [31]:
# group_average_df
# 

In [32]:
# gpt_human_performance_df

In [33]:
gpt_human_performance_df = group_average_df[group_average_df['agent_type'].isin(['GPT-4V', 'Human'])]
gpt_human_performance_df = gpt_human_performance_df.groupby(
    ['agent_type', 'item_id', 'task_category', 'chart_type']
)['is_correct'].mean().reset_index()

In [34]:
# gpt_human_performance_df

In [35]:
%%R -i gpt_human_performance_df

null_model <- lmer(
    is_correct ~ 1 + agent_type + (1 | item_id), 
    data = gpt_human_performance_df,
)

graph_model <- lmer(
    is_correct ~ 1 + chart_type * agent_type + (1 | item_id), 
    data = gpt_human_performance_df, 
)

anova(null_model, graph_model, test="LRT")

Data: gpt_human_performance_df
Models:
null_model: is_correct ~ 1 + agent_type + (1 | item_id)
graph_model: is_correct ~ 1 + chart_type * agent_type + (1 | item_id)
            npar    AIC    BIC  logLik deviance  Chisq Df Pr(>Chisq)    
null_model     4 1273.3 1294.7 -632.65   1265.3                         
graph_model   28 1218.8 1368.8 -581.37   1162.8 102.55 24  1.101e-11 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


refitting model(s) with ML (instead of REML)


In [36]:
%%R -i gpt_human_performance_df

null_model <- lmer(
    is_correct ~ 1 + agent_type + (1 | item_id), 
    data = gpt_human_performance_df,
)

graph_model <- lmer(
    is_correct ~ 1 + task_category * agent_type + (1 | item_id), 
    data = gpt_human_performance_df, 
)

anova(null_model, graph_model, test="LRT")

Data: gpt_human_performance_df
Models:
null_model: is_correct ~ 1 + agent_type + (1 | item_id)
graph_model: is_correct ~ 1 + task_category * agent_type + (1 | item_id)
            npar    AIC    BIC  logLik deviance  Chisq Df Pr(>Chisq)    
null_model     4 1273.3 1294.7 -632.65   1265.3                         
graph_model    8 1052.5 1095.4 -518.26   1036.5 228.77  4  < 2.2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


refitting model(s) with ML (instead of REML)


In [37]:
gpt_human_graph_performance_df = gpt_human_performance_df.pivot(
    index=['item_id', 'chart_type'], columns='agent_type', values='is_correct'
).reset_index().dropna()
gpt_human_graph_performance_df['human_gpt_gap'] = gpt_human_graph_performance_df['GPT-4V'] - gpt_human_graph_performance_df['Human']

gpt_human_task_performance_df = gpt_human_performance_df.pivot(
    index=['item_id', 'task_category'], columns='agent_type', values='is_correct'
).reset_index().dropna()
gpt_human_task_performance_df['human_gpt_gap'] = gpt_human_task_performance_df['GPT-4V'] - gpt_human_task_performance_df['Human']

In [39]:
# graph_corr_vector = gpt_human_graph_performance_df.groupby(['chart_type', '']).agg(
    
# )

### Graph Correlation

In [40]:
np.corrcoef(gpt_human_graph_performance_df['Human'].to_numpy(), gpt_human_graph_performance_df['GPT-4V'].to_numpy())

array([[1.        , 0.60059023],
       [0.60059023, 1.        ]])

#### Task Correlation

In [41]:
np.corrcoef(gpt_human_task_performance_df['Human'].to_numpy(), gpt_human_task_performance_df['GPT-4V'].to_numpy())

array([[1.        , 0.60059023],
       [0.60059023, 1.        ]])